# Willow biochar on Swedish arable land: spatial and time-explicit LCA

Code for *Climate impact of short-rotation coppice willow converted to biochar for soil
application in Sweden: a spatially resolved, time-explicit assessment* (Van Rysselberge,
Ericsson, Karlsson, Sundberg, Hansson).

The notebook goes from the files in `data/` to the results in the paper:

1. Settings
2. Brightway project and ecoinvent
3. Soil organic carbon (ICBM spin-up and 100 years of willow)
4. Inventory inputs per harvest zone
5. Climate metric functions
6. Foreground inventory, one activity per zone and year
7. Temperature response across zones
8. GWP100 per zone
9. Contributions in the reference zone
10. Other scenarios
11. Scenario comparison
12. Monte Carlo and Sobol indices

Run it from top to bottom. Tables are written to `outputs/` and figures to
`outputs/figures/`. Installation and the ecoinvent requirement are described in `README.md`.

## 1. Settings

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Scenario built and analysed first; the others are built in section 10.
SCENARIO = "high_yield"

PROJECT_NAME = "willow_biochar"
ECOINVENT_DB = "cutoff391"   # the inventory template refers to this database name
# Folder with the ecoinvent 3.9.1 cut-off ecospold files, only used for the first import
ECOINVENT_DIR = r"C:/path/to/ecoinvent 3.9.1_cutoff_ecoSpold02/datasets"

NOTEBOOK_FILE = "willow_biochar_lca.ipynb"   # section 10 re-reads tagged cells from this file

DATA_DIR = "data"
OUT_DIR = "outputs"
RASTER_DIR = os.path.join(OUT_DIR, "rasters")
FIG_DIR = os.path.join(OUT_DIR, "figures")
for d in (OUT_DIR, RASTER_DIR, FIG_DIR):
    os.makedirs(d, exist_ok=True)

ZONES_SHP = os.path.join(DATA_DIR, "harvest_zones_sweref99.shp")
STUDIED_MAX = 2122          # harvest zones with codes above this lie north of the willow limit
REFERENCE_ZONE = "1821"     # zone whose result equals the across-zone median

# Heat from pyrolysis displaces district heating (kg CO2-eq per kWh delivered)
DH_SWEDISH_MIX = 0.0553     # Swedish average 2024, Energiforetagen (2025)
DH_FOSSIL = 0.199           # ecoinvent 3.9.1, market for heat, district or industrial, natural gas

# ICBM parameters, used for both the spin-up and the willow phase (Hammar et al. 2014)
ICBM_PARAMS = dict(kY=0.80, kO=0.0085, ha=0.17, hb=0.39, re=1.0)

# Harvest schedule, shared by the ICBM inputs and the inventory.
# 25-year plantations, harvests in years 3, 6, ..., 24, stools removed in year 25.
# The first harvest of a plantation is 60 % of a later one. With the zone yield Y
# as the 100-year mean, a later harvest is Y * 100 / (4 * (0.6 + 7)) = 3.29 Y.
FIRST_HARVEST_FACTOR = 0.60
HARVESTS_PER_PLANTATION = 8
LATER_HARVEST_PER_YIELD = 100.0 / (4 * (FIRST_HARVEST_FACTOR + HARVESTS_PER_PLANTATION - 1))

# Willow litter and root inputs to ICBM, Hammar et al. (2014, Table 2): leaves, fine
# roots and coarse roots as ratios to the stem growth of the same year, 0.5 kg C/kg DM.
# Coarse roots and stumps enter the below-ground pool when the stools are removed.
WILLOW_C_FRAC = 0.50
STEM_SHARE = (0.25, 0.40, 0.35)        # stem growth in years 1-3 of a cutting cycle
LEAF_1ST = (0.25, 0.30, 0.27)          # first cutting cycle
FINE_ROOT_1ST = (0.60, 0.71, 0.70)
COARSE_1ST = (0.39, 0.19, 0.12)
LEAF_LATER, FINE_ROOT_LATER = 0.31, 0.69


def salix_input_fractions(year):
    """Above- and below-ground C input in a given year, t C per t DM/ha/yr of zone yield."""
    yic = (year - 1) % 25 + 1
    later = LATER_HARVEST_PER_YIELD
    first = FIRST_HARVEST_FACTOR * later
    if yic == 25:
        coarse = sum(first * s * c for s, c in zip(STEM_SHARE, COARSE_1ST))
        return 0.0, WILLOW_C_FRAC * coarse
    k = (yic - 1) % 3
    if yic <= 3:
        stem = first * STEM_SHARE[k]
        return WILLOW_C_FRAC * stem * LEAF_1ST[k], WILLOW_C_FRAC * stem * FINE_ROOT_1ST[k]
    stem = later * STEM_SHARE[k]
    return WILLOW_C_FRAC * stem * LEAF_LATER, WILLOW_C_FRAC * stem * FINE_ROOT_LATER


# Biochar conversion, Leppakoski et al. (2021)
BIOCHAR_MASS_YIELD = 0.35                 # kg dry biochar per kg dry chips
DRYING_DM_RETAINED = 2857.14 / 2945.51    # dry matter left after storage in piles
BIOCHAR_C_CONTENT = 0.72                  # kg C per kg biochar
HANDLING_RETENTION = 0.95                 # 5 % lost in handling, transport and spreading
BIOCHAR_YIELD_ON_HARVEST = BIOCHAR_MASS_YIELD * DRYING_DM_RETAINED                   # 0.3395
CO2_PER_KG_BIOCHAR_TO_SOIL = BIOCHAR_C_CONTENT * HANDLING_RETENTION * 44.0 / 12.0    # 2.508

SCENARIOS = {
    "high_yield":            dict(yield_raster="yield_potential.tif",    durable=(85.0, 95.0), dh_factor=DH_SWEDISH_MIX),
    "low_yield":             dict(yield_raster="yield_top_quartile.tif", durable=(85.0, 95.0), dh_factor=DH_SWEDISH_MIX),
    "low_biochar_stability": dict(yield_raster="yield_potential.tif",    durable=(50.0, 70.0), dh_factor=DH_SWEDISH_MIX),
    "fossil_heat":           dict(yield_raster="yield_potential.tif",    durable=(85.0, 95.0), dh_factor=DH_FOSSIL),
}


def scenario_db_name(scenario):
    return f"willow_{scenario}"


SCEN = SCENARIOS[SCENARIO]
YIELD_RASTER = os.path.join(DATA_DIR, SCEN["yield_raster"])
DATABASE_NAME = scenario_db_name(SCENARIO)
# ICBM output depends only on the yield raster, so it is named after it
SALIX_ICBM_XLSX = os.path.join(OUT_DIR, f"soc_willow_{os.path.splitext(SCEN['yield_raster'])[0]}.xlsx")


def zone_valid_mean(values):
    """Mean over the cultivated pixels of a zone (yield > 0), the pixels ICBM simulates."""
    v = np.asarray(values, dtype=float)
    v = v[np.isfinite(v) & (v > 0)]
    return float(v.mean()) if v.size else np.nan


def write_database(name, data):
    """Replace a Brightway database. Deleting first avoids a VACUUM-in-transaction
    error in bw2data when an existing database with >500 activities is overwritten."""
    import brightway2 as bw2
    if name in bw2.databases:
        del bw2.databases[name]
    bw2.Database(name).write(data)


FIG_W = 7.0
FIG_SIZE = (FIG_W, 4.5)
FIG_MAP = (FIG_W, FIG_W * 1366 / 1113)
FIG_WIDE = (FIG_W * 1.6, 4.5)
plt.rcParams.update({
    "figure.figsize": FIG_SIZE, "font.size": 10, "axes.labelsize": 10,
    "xtick.labelsize": 10, "ytick.labelsize": 10, "legend.fontsize": 10,
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
})

print(f"Scenario {SCENARIO}: database {DATABASE_NAME}, yield {YIELD_RASTER}, "
      f"BC100 band {SCEN['durable']}, displaced heat {SCEN['dh_factor']} kg CO2-eq/kWh")

## 2. Brightway project and ecoinvent

ecoinvent 3.9.1 (cut-off) is imported once into the project under the name `cutoff391`.
Set `ECOINVENT_DIR` above before the first run.

In [ ]:
import pandas as pd
import brightway2 as bw2
import bw2io as bi

bw2.projects.set_current(PROJECT_NAME)
if "biosphere3" not in bw2.databases:
    bi.bw2setup()
if ECOINVENT_DB not in bw2.databases:
    ei = bi.SingleOutputEcospold2Importer(ECOINVENT_DIR, ECOINVENT_DB)
    ei.apply_strategies()
    ei.match_database(fields=("name", "unit", "location"))
    ei.statistics()
    ei.write_database()

GWP_METHOD = ("IPCC 2013", "climate change", "global warming potential (GWP100)")
assert GWP_METHOD in bw2.methods
print(sorted(bw2.databases))

## 3. Soil organic carbon

The previous annual cropping is run to steady state for each zone (spin-up), the pools
are put on the model grid, and willow is then simulated for 100 years on every pixel with
a positive yield. ICBM follows Andrén et al. (2004) with separate young pools for above-
and below-ground inputs.

In [ ]:
# Spin-up: 1000 years of the previous cropping, zone-mean inputs Ia and Ib (t C/ha/yr).
# Humified carbon is taken from the young pool after input and before decay.
SPINUP_YEARS = 1000
SPINUP_POOLS = os.path.join(OUT_DIR, "spinup_pools.xlsx")

spin_in = pd.read_excel(os.path.join(DATA_DIR, "spinup_inputs.xlsx"))
ia = spin_in["Ia"].to_numpy(dtype=float)
ib = spin_in["Ib"].to_numpy(dtype=float)
p = ICBM_PARAMS
eY, eO = np.exp(-p["kY"] * p["re"]), np.exp(-p["kO"] * p["re"])

Ya = np.zeros_like(ia); Yb = np.zeros_like(ib)
Oa = np.zeros_like(ia); Ob = np.zeros_like(ib)
for _ in range(SPINUP_YEARS):
    Ya_in, Yb_in = Ya + ia, Yb + ib
    Oa = (Oa + p["ha"] * Ya_in * (1 - eY)) * eO
    Ob = (Ob + p["hb"] * Yb_in * (1 - eY)) * eO
    Ya, Yb = Ya_in * eY, Yb_in * eY

spinup = pd.DataFrame({"zone": spin_in["Region"], "Ya": Ya, "Yb": Yb, "O_above": Oa, "O_below": Ob})
spinup["total"] = Ya + Yb + Oa + Ob
spinup.to_excel(SPINUP_POOLS, index=False)

# old pools should match the analytical steady state h*i/(re*kO)
print("O / steady state, above:", round(np.mean(Oa / (p["ha"] * ia / (p["re"] * p["kO"]))), 4),
      "below:", round(np.mean(Ob / (p["hb"] * ib / (p["re"] * p["kO"]))), 4))
print(f"Initial SOC: median {spinup.total.median():.1f}, "
      f"range {spinup.total.min():.1f}-{spinup.total.max():.1f} t C/ha")

In [ ]:
# Put the spin-up pools on the model grid
import geopandas as gpd
import rasterio
from rasterio.features import rasterize

with rasterio.open(os.path.join(DATA_DIR, "model_grid.tif")) as src:
    grid_meta = src.meta.copy()
    grid_crs, grid_shape, grid_tf = src.crs, (src.height, src.width), src.transform
grid_meta.update(dtype="float32", count=1)

_zones = gpd.read_file(ZONES_SHP).to_crs(grid_crs)
pools = {int(r.zone): (r.Ya, r.Yb, r.O_above + r.O_below) for r in spinup.itertuples()}
missing = sorted({int(z) for z in _zones["SKOMR"]} - set(pools))
assert not missing, f"no spin-up result for zones {missing}"

for name, idx in (("pool_Ya", 0), ("pool_Yb", 1), ("pool_O", 2)):
    arr = rasterize([(g, float(pools[int(z)][idx])) for g, z in zip(_zones.geometry, _zones["SKOMR"])],
                    out_shape=grid_shape, transform=grid_tf, fill=0.0, all_touched=False, dtype="float32")
    with rasterio.open(os.path.join(RASTER_DIR, f"{name}.tif"), "w", **grid_meta) as dst:
        dst.write(arr, 1)
print("Pool rasters written to", RASTER_DIR)

In [ ]:
# Willow, 100 years, on every cultivated pixel
from rasterio.features import geometry_mask
from rasterio.warp import reproject, Resampling

kY, kO, ha, hb, re_factor = (ICBM_PARAMS[k] for k in ("kY", "kO", "ha", "hb", "re"))


def load_raster(path):
    with rasterio.open(path) as src:
        return src.read(1).astype(float), src.nodata


with rasterio.open(os.path.join(RASTER_DIR, "pool_Ya.tif")) as src:
    Ya, ya_nd = src.read(1).astype(float), src.nodata
    transform, raster_height, raster_width, master_crs = src.transform, src.height, src.width, src.crs
Yb, yb_nd = load_raster(os.path.join(RASTER_DIR, "pool_Yb.tif"))
O, o_nd = load_raster(os.path.join(RASTER_DIR, "pool_O.tif"))

# yield raster onto the pool grid
with rasterio.open(YIELD_RASTER) as src:
    src_yield, src_nd, src_crs, src_tf = src.read(1).astype(float), src.nodata, src.crs, src.transform
yield_raster = np.full((raster_height, raster_width), np.nan)
reproject(source=src_yield, destination=yield_raster,
          src_transform=src_tf, src_crs=src_crs, dst_transform=transform, dst_crs=master_crs,
          src_nodata=src_nd, dst_nodata=np.nan, resampling=Resampling.nearest)

zones = gpd.read_file(ZONES_SHP).to_crs(master_crs)


def valid(arr, nodata, drop_zero):
    m = np.isfinite(arr)
    if nodata is not None:
        m &= arr != nodata
    if drop_zero:
        m &= arr != 0
    return m


# study area: positive yield, valid pools, inside a studied zone (pixel centre)
study_mask = valid(yield_raster, None, True) & (yield_raster > 0)
study_mask &= valid(Ya, ya_nd, False) & valid(Yb, yb_nd, False) & valid(O, o_nd, False)
studied = zones[zones["SKOMR"].astype(int) <= STUDIED_MAX]
study_mask &= rasterize([(g, 1) for g in studied.geometry], out_shape=(raster_height, raster_width),
                        transform=transform, fill=0, all_touched=False, dtype=np.uint8).astype(bool)
print("Study-area pixels:", int(study_mask.sum()))

yield_raster = np.where(study_mask, yield_raster, np.nan)
Ya, Yb, O = (np.where(study_mask, a, np.nan) for a in (Ya, Yb, O))


def compute_next_year(Ya, Yb, O, i_ag, i_bg, re_factor):
    e = np.exp(-kY * re_factor)
    Ya_next = (Ya + i_ag) * e
    Yb_next = (Yb + i_bg) * e
    O_next = (O + (hb * (Yb + i_bg) + ha * (Ya + i_ag)) * (1 - e)) * np.exp(-kO * re_factor)
    return Ya_next, Yb_next, O_next


def calculate_inputs_salix(year, yield_raster):
    f_ag, f_bg = salix_input_fractions(year)
    return yield_raster * f_ag, yield_raster * f_bg


zone_masks = {str(r.SKOMR): geometry_mask([r.geometry], invert=True, transform=transform,
                                          out_shape=(raster_height, raster_width))
              for r in zones.itertuples()}


def zonal_mean(total):
    out = {}
    for z, m in zone_masks.items():
        v = total[m]
        out[z] = np.nanmean(v) if np.isfinite(v).any() else np.nan
    return out


total_results = {0: Ya + Yb + O}
zone_results = {z: {0: v} for z, v in zonal_mean(total_results[0]).items()}
for year in range(1, 101):
    i_ag, i_bg = calculate_inputs_salix(year, yield_raster)
    Ya, Yb, O = compute_next_year(Ya, Yb, O, i_ag, i_bg, re_factor)
    total_results[year] = Ya + Yb + O
    for z, v in zonal_mean(total_results[year]).items():
        zone_results[z][year] = v

luc_df = pd.DataFrame(zone_results).T.sort_index()
luc_df.index.name = "Zone"
luc_df.columns = [f"Year_{c}" for c in luc_df.columns]
luc_df.to_excel(SALIX_ICBM_XLSX)
print("Saved", SALIX_ICBM_XLSX)

# bounding box of the study area, for the maps
finite = np.isfinite(total_results[100])
rows, cols = np.where(finite.any(axis=1))[0], np.where(finite.any(axis=0))[0]
r0, r1 = max(rows[0] - 5, 0), min(rows[-1] + 6, raster_height)
c0, c1 = max(cols[0] - 5, 0), min(cols[-1] + 6, raster_width)

In [ ]:
# SOC under willow across zones (median and percentile bands)
soc = np.array([[zone_results[z].get(y, np.nan) for y in range(101)]
                for z in zone_results if int(z) <= STUDIED_MAX])
soc = soc[~np.all(np.isnan(soc), axis=1)]
years = np.arange(101)
pc = {q: np.nanpercentile(soc, q, axis=0) for q in (5, 25, 50, 75, 95)}

plt.figure(figsize=FIG_SIZE)
plt.fill_between(years, pc[5], pc[95], alpha=0.15, label="5th–95th percentile range across zones")
plt.fill_between(years, pc[25], pc[75], alpha=0.30, label="25th–75th percentile range across zones")
plt.plot(years, pc[50], lw=2.5, label="Median across zones")
plt.xlabel("Years after plantation establishment")
plt.ylabel("Soil organic carbon (t C ha$^{-1}$)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig(os.path.join(FIG_DIR, "fig05_soc_trajectory.png"))
plt.show()

gain = soc[:, -1] - soc[:, 0]
print(f"{soc.shape[0]} zones. Median SOC {pc[50][0]:.1f} -> {pc[50][-1]:.1f} t C/ha; "
      f"per-zone gain {gain.min():.1f} to {gain.max():.1f}, median {np.median(gain):.1f} t C/ha")

In [ ]:
# Change in SOC between year 0 and year 100
diff = total_results[100] - total_results[0]
show = np.ma.masked_invalid(diff)[r0:r1, c0:c1]
vmax = np.nanmax(np.abs(show))

fig = plt.figure(figsize=FIG_MAP)
ax = fig.add_axes([0.075, 0.175, 0.65, 0.65])
im = ax.imshow(show, cmap="BrBG", vmin=-vmax, vmax=vmax)
ax.axis("off")
fig.colorbar(im, cax=fig.add_axes([0.84, 0.20, 0.035, 0.60]), label="ΔSOC (t C ha$^{-1}$)")
plt.savefig(os.path.join(FIG_DIR, "fig06_soc_change_map.png"), pad_inches=0.05)
plt.show()

v = diff[np.isfinite(diff)]
print(f"ΔSOC per pixel: mean {v.mean():.1f}, min {v.min():.1f}, max {v.max():.1f} t C/ha; "
      f"{(v > 0).mean():.1%} of pixels gain carbon")

## 4. Inventory inputs per harvest zone

Yield-dependent flows (nitrogen fertiliser, chip transport, harrowing and soil N₂O) are
the zone yield times the per-tonne factors of the inventory template (totals over 100 years
per t DM ha⁻¹ yr⁻¹, Table S6). Zone means use the same pixels as ICBM.

In [ ]:
FACTORS = {
    "Nitrogen":          ("Nitrogen_production.tif", 579.0688726),
    "Vägtransport":      ("Vagtransport.tif", 19764.142582),   # 60 km haul of chips
    "Harvning":          ("Harvning.tif", 114.0521739),
    "Soil_N2O_direct":   ("Soil_N2O_direct.tif", 9),
    "Soil_N2O_indirect": ("Soil_N2O_indirect.tif", 2),
    "Soil_N2O_litter":   ("Soil_N2O_litter.tif", 21),
}

with rasterio.open(YIELD_RASTER) as src:
    yield_data, meta = src.read(1), src.meta.copy()
meta.update(dtype=rasterio.float32)

raster_paths = {}
for key, (fname, factor) in FACTORS.items():
    path = os.path.join(RASTER_DIR, fname)
    with rasterio.open(path, "w", **meta) as dst:
        dst.write((yield_data * factor).astype(rasterio.float32), 1)
    raster_paths[key] = path

zone_means = {k: {} for k in raster_paths}
for key, path in raster_paths.items():
    with rasterio.open(path) as src:
        data = src.read(1)
        for _, row in zones.iterrows():
            m = geometry_mask([row.geometry], invert=True, transform=src.transform, out_shape=data.shape)
            zone_means[key][row["SKOMR"]] = zone_valid_mean(data[m])
print("Zone means of", ", ".join(zone_means))

In [ ]:
with rasterio.open(YIELD_RASTER) as src:
    data, nodata = src.read(1), src.nodata
    zone_means_yield = {}
    for _, row in zones.iterrows():
        m = geometry_mask([row.geometry], invert=True, transform=src.transform, out_shape=(src.height, src.width))
        v = data[m]
        if nodata is not None:
            v = v[v != nodata]
        zone_means_yield[row["SKOMR"]] = zone_valid_mean(v)

y = pd.Series({str(int(z)): v for z, v in zone_means_yield.items()})
print(f"Zone mean yield {y.min():.2f}-{y.max():.2f} t DM/ha/yr; "
      f"reference zone {REFERENCE_ZONE}: {y[REFERENCE_ZONE]:.2f}")

## 5. Climate metric functions

Temperature response per kg of CO₂, CH₄ and N₂O (AGTP, Collins et al. 2013, AR5 constants),
and a routine that splits the CO₂, CH₄ and N₂O of each foreground activity into its own
emissions and the upstream (ecoinvent) emissions of its inputs.

In [ ]:
def generate_temp_responses(n=100, vers="AR5", model="C2012"):
    """Temperature response per kg emitted (K/kg) for CO2, CH4 and N2O over n+1 years.

    Written by Niclas Ericsson.
    vers  : "AR5" or "AR4" constants
    model : "C2012" (Collins et al., AGTP), "E2013" (Ericsson et al. 2013, convolution)
            or "E2014" (Ericsson et al. 2014, fitted constants)
    Returns {"co2": array, "ch4": array, "n2o": array}.
    """
    from copy import deepcopy

    cs = deepcopy({"c1": 0.631, "c2": 0.429, "d1": 8.4, "d2": 409.5})
    if vers == "AR5":
        cs.update({
            "a0": 0.2173, "a1": 0.2240, "a2": 0.2824, "a3": 0.2763,
            "t1co2": 394.4, "t2co2": 36.54, "t3co2": 4.304,
            "tch4": 12.4, "tn2o": 121.0,
            "f1": 0.5, "f2": 0.15,
            "M0": 1803, "M": 1804, "N0": 324, "N": 325, "C0": 391, "C": 392,
            "AlphaM": 0.036, "AlphaN": 0.12, "AlphaC": 5.35,
            "Ma": 28.97, "MxM": 16.04276, "MxN": 44.01288, "MxC": 44.0098,
            "Tm": 5.1352e18,
        })
    elif vers == "AR4":
        cs.update({
            "a0": 0.217, "a1": 0.259, "a2": 0.338, "a3": 0.186,
            "t1co2": 172.9, "t2co2": 18.51, "t3co2": 1.186,
            "tch4": 12.0, "tn2o": 114.0,
            "f1": 0.25, "f2": 0.15,
            "M0": 1774, "M": 1775, "N0": 319, "N": 320, "C0": 378, "C": 379,
            "AlphaM": 0.036, "AlphaN": 0.12, "AlphaC": 5.35,
            "Ma": 28.96, "MxM": 16.04276, "MxN": 44.01288, "MxC": 44.0098,
            "Tm": 5.12e18,
        })
    else:
        raise ValueError("Unsupported version specified")

    t = np.arange(0, n + 1)

    # CH4-N2O band overlap (IPCC TAR)
    f_mn0 = 0.47 * np.log(1 + 2.01e-5 * (cs["M"] * cs["N0"]) ** 0.75 + 5.31e-15 * cs["M"] * (cs["M"] * cs["N0"]) ** 1.52)
    f_m0n0 = 0.47 * np.log(1 + 2.01e-5 * (cs["M0"] * cs["N0"]) ** 0.75 + 5.31e-15 * cs["M0"] * (cs["M0"] * cs["N0"]) ** 1.52)
    f_m0n = 0.47 * np.log(1 + 2.01e-5 * (cs["M0"] * cs["N"]) ** 0.75 + 5.31e-15 * cs["M0"] * (cs["M0"] * cs["N"]) ** 1.52)

    # radiative efficiencies (W/m2) for a 1 ppb/ppm increase, then per kg
    re_ch4v = cs["AlphaM"] * (np.sqrt(cs["M"]) - np.sqrt(cs["M0"])) - (f_mn0 - f_m0n0)
    re_n2ov = cs["AlphaN"] * (np.sqrt(cs["N"]) - np.sqrt(cs["N0"])) - (f_m0n - f_m0n0)
    re_co2v = cs["AlphaC"] * np.log(cs["C"] / cs["C0"])
    f_ch4 = cs["Tm"] / 1e9 / cs["Ma"] * cs["MxM"] * (cs["M"] - cs["M0"])
    f_n2o = cs["Tm"] / 1e9 / cs["Ma"] * cs["MxN"] * (cs["N"] - cs["N0"])
    f_co2 = cs["Tm"] / 1e6 / cs["Ma"] * cs["MxC"] * (cs["C"] - cs["C0"])
    re_ch4 = re_ch4v * (1 + cs["f1"] + cs["f2"]) / f_ch4
    re_n2o = re_n2ov / f_n2o
    re_co2 = re_co2v / f_co2

    if model == "E2013":
        c_co2 = cs["a0"] + cs["a1"] * np.exp(-t / cs["t1co2"]) + cs["a2"] * np.exp(-t / cs["t2co2"]) + cs["a3"] * np.exp(-t / cs["t3co2"])
        rt = cs["c1"] / cs["d1"] * np.exp(-t / cs["d1"]) + cs["c2"] / cs["d2"] * np.exp(-t / cs["d2"])
        dt_co2 = np.convolve(rt, re_co2 * c_co2)[:n + 1]
        dt_ch4 = np.convolve(rt, re_ch4 * np.exp(-t / cs["tch4"]))[:n + 1]
        dt_n2o = np.convolve(rt, re_n2o * np.exp(-t / cs["tn2o"]))[:n + 1]

    elif model == "E2014":
        dt_co2 = re_co2 * (0.23 - 0.02 * np.exp(-t / 1.186) - 0.68 * np.exp(-t / 8.4) + 0.384 * np.exp(-t / 18.51)
                           + 0.091 * np.exp(-t / 172.9) - 0.005 * np.exp(-t / 409.5))
        dt_ch4 = re_ch4 * (-1.984 * np.exp(-t / 8.4) + 1.972 * np.exp(-t / 12) + 0.012 * np.exp(-t / 409.5))
        dt_n2o = re_n2o * (-0.643 * np.exp(-t / 8.4) + 0.487 * np.exp(-t / 114) + 0.156 * np.exp(-t / 409.5))

    elif model == "C2012":
        # Collins et al. (2013), eq. A3: each CO2 term pairs with both climate modes (c1, d1) and (c2, d2)
        def persistent(c, d):
            return cs["a0"] * c * (1 - np.exp(-t / d))

        def decaying(a, tau, c, d):
            return a * tau * c / (tau - d) * (np.exp(-t / tau) - np.exp(-t / d))

        dt_co2 = re_co2 * sum(
            persistent(c, d) + sum(decaying(cs[a], cs[tau], c, d)
                                   for a, tau in (("a1", "t1co2"), ("a2", "t2co2"), ("a3", "t3co2")))
            for c, d in ((cs["c1"], cs["d1"]), (cs["c2"], cs["d2"])))
        dt_ch4 = re_ch4 * (decaying(1, cs["tch4"], cs["c1"], cs["d1"]) + decaying(1, cs["tch4"], cs["c2"], cs["d2"]))
        dt_n2o = re_n2o * (decaying(1, cs["tn2o"], cs["c1"], cs["d1"]) + decaying(1, cs["tn2o"], cs["c2"], cs["d2"]))
    else:
        raise ValueError("Unsupported model")

    return {"co2": dt_co2, "ch4": dt_ch4, "n2o": dt_n2o}


def convolve_pulses_with_kernel(pulses, kernel, horizon):
    out = np.zeros(horizon + 1)
    for year, amount in enumerate(pulses):
        if amount:
            out[year:] += amount * kernel[:horizon + 1 - year]
    return out


# Flows with a non-zero IPCC 2013 GWP100 factor; uptake into storage counts as negative.
GAS_FLOW_NAMES = {
    "Carbon dioxide, fossil":                     ("co2", 1.0),
    "Carbon dioxide, from soil or biomass stock": ("co2", 1.0),
    "Carbon dioxide, to soil or biomass stock":   ("co2", -1.0),
    "Methane, fossil":                            ("ch4", 1.0),
    "Methane, non-fossil":                        ("ch4", 1.0),
    "Methane, from soil or biomass stock":        ("ch4", 1.0),
    "Dinitrogen monoxide":                        ("n2o", 1.0),
}
GASES = ("co2", "ch4", "n2o")


def gas_of_flow(flow_key):
    return GAS_FLOW_NAMES.get(bw2.get_activity(flow_key)["name"], (None, 0.0))


def lca_index_maps(lca):
    """Biosphere, activity and product index dictionaries (bw2calc 1.x and 2.x)."""
    if hasattr(lca, "dicts") and hasattr(lca.dicts, "biosphere"):
        return dict(lca.dicts.biosphere), dict(lca.dicts.activity), dict(lca.dicts.product)
    return lca.biosphere_dict, lca.activity_dict, lca.product_dict


def gas_per_unit_product(lca):
    """kg of each gas per unit of every product, cradle to gate: solve A^T y = B^T c."""
    import scipy.sparse.linalg as spla
    bio_row, _, _ = lca_index_maps(lca)
    c = {g: np.zeros(lca.biosphere_matrix.shape[0]) for g in GASES}
    for flow, row in bio_row.items():
        gas, sign = gas_of_flow(flow)
        if gas:
            c[gas][row] = sign
    lu = spla.splu(lca.technosphere_matrix.T.tocsc())
    return {g: lu.solve(np.asarray(lca.biosphere_matrix.T @ c[g]).ravel()) for g in GASES}


def zone_year_gas_inventory(activities, fu_amount=1.0):
    """CO2, CH4 and N2O per (activity, zone, year): the activity's own exchanges
    ("foreground") and the upstream emissions of its inputs ("background"),
    both assigned to the year of the activity."""
    lca = bw2.LCA({a.key: fu_amount for a, _, _ in activities})
    lca.lci()
    bio_row, act_col, prod_row = lca_index_maps(lca)
    y = gas_per_unit_product(lca)
    gas_row = np.full(lca.biosphere_matrix.shape[0], "", dtype=object)
    sign_row = np.zeros(lca.biosphere_matrix.shape[0])
    for flow, row in bio_row.items():
        gas, sign = gas_of_flow(flow)
        if gas:
            gas_row[row], sign_row[row] = gas, sign
    B, A = lca.biosphere_matrix.tocsc(), lca.technosphere_matrix.tocsc()
    rows = []
    for act, zone, year in activities:
        col, own = act_col[act.key], prod_row[act.key]
        b = B.getcol(col).tocoo()
        fg = dict.fromkeys(GASES, 0.0)
        for r, v in zip(b.row, b.data):
            if gas_row[r]:
                fg[gas_row[r]] += sign_row[r] * v * fu_amount
        a = A.getcol(col).tocoo()
        keep = a.row != own
        amounts = -a.data[keep] * fu_amount
        for g in GASES:
            rows.append((zone, year, g, "foreground", fg[g]))
            rows.append((zone, year, g, "background", float(amounts @ y[g][a.row[keep]])))
    return pd.DataFrame(rows, columns=["zone", "year", "flow", "source", "amount_kg"])


def grouped_tag(tag, name=""):
    """Component of an exchange in the temperature contribution figure."""
    t = str(tag).lower() if tag is not None else ""
    n = str(name).lower() if name is not None else ""
    s = f"{t} | {n}"
    if tag == "LUC_SOC_willow":
        return "Soil organic carbon"
    if tag == "avoided_district_heating":
        return "Avoided emissions from heat substitution"
    if tag in {"Biochar", "biochar_CO2_decay"}:
        return "Biochar"
    if "biochar" in s or "electricity, medium voltage" in s or "tap water" in s:
        return "Biochar"
    return "Willow production"

## 6. Foreground inventory

One Brightway activity per harvest zone and year (years 0–100). Annual entries are
end-of-year pulses and year 0 carries no flows. Establishment falls in year 1 of each
25-year plantation, harvests in years 3, 6, …, 24 and stool removal in year 25. Flows that
occur every year are spread over years 1–100.

In harvest years the post-harvest chain of Leppäkoski et al. (2021) is added: storage and
natural drying of the rods, roadside chipping, transport, drying, slow pyrolysis,
moistening, transport and spreading of the biochar, and the credit for exported heat. The
storage of 9–18 months before pyrolysis is collapsed into the harvest year.

Biochar carbon enters storage in the harvest year and decays from the following year with
the mean of the Azzi et al. (2024) strategy S2 fits inside the scenario's BC₁₀₀ band.

In [ ]:
import re

EXCEL_TEMPLATE = os.path.join(DATA_DIR, "inventory_template.xlsx")
DECAY_FITS_XLSX = os.path.join(DATA_DIR, "biochar_decay_fits_Azzi2024.xlsx")
DURABLE_MIN, DURABLE_MAX = SCEN["durable"]
MAX_DIFF_S2_S3 = 20.0          # quality screen of Azzi et al. (2024)
YEARS = np.arange(0, 101)
YEAR_FACTOR = 1 / 100          # yearly flows over years 1-100
DH_FACTOR = SCEN["dh_factor"]

# Chain per kg of dry biochar produced, Leppakoski et al. (2021), Table A1
BIOCHAR_CHAIN = {
    "diesel_chipping_kg": 7.63 / 1000,
    "transport_distance_km": 60.0,
    "electricity_artificial_drying_kwh": 48.05 / 1000,
    "electricity_pyrolysis_kwh": 400.0 / 1000,     # 0.126 kWh/kg x 3174.6 kg dried chips
    "avoided_district_heat_kwh": 3422.25 / 1000,   # exported heat; 858.69 kWh/t is used for drying
    "water_moistening_kg": 176.47 / 1000,
    "wet_biochar_15_out": 1176.47 / 1000,
    "diesel_soil_application_kg": 0.81 / 1000,
}
# The ecoinvent diesel activity is per MJ and includes combustion: 0.02222 kg diesel per MJ
DIESEL_MJ_PER_KG = 1.0 / 0.02222


def norm_zone(z):
    return str(int(str(z).strip()))


def safe_number(x, default=0.0):
    try:
        x = float(x)
        return default if np.isnan(x) else x
    except (TypeError, ValueError):
        return default


# ---- biochar decay --------------------------------------------------------
def eval_s2_curve(row, years):
    """Percentage of biochar C remaining after `years`, from one fitted model."""
    t = years * 365.25
    num = lambda k: pd.to_numeric(row.get(k), errors="coerce")
    model = str(row["strategy_model"])
    if "doubleExp" in model:
        c1, c2 = num("parameter_c1"), num("parameter_c2")
        c2 = 100.0 - c1 if np.isnan(c2) else c2
        return c1 * np.exp(-num("parameter_k1") * t) + c2 * np.exp(-num("parameter_k2") * t)
    if "tripleExp" in model:
        c1, c2, c3 = num("parameter_c1"), num("parameter_c2"), num("parameter_c3")
        c3 = 100.0 - c1 - c2 if np.isnan(c3) else c3
        return (c1 * np.exp(-num("parameter_k1") * t) + c2 * np.exp(-num("parameter_k2") * t)
                + c3 * np.exp(-num("parameter_k3") * t))
    if "singleExp" in model:
        c = num("parameter_c")
        return (100.0 if np.isnan(c) else c) * np.exp(-num("parameter_k") * t)
    raise ValueError(f"Unsupported model: {model}")


def load_decay_fits(path, band):
    """Strategy S2 fits within a BC100 band, after the screening of Azzi et al. (2024)."""
    raw = pd.read_excel(path, header=[0, 1])
    raw.columns = [f"{a}_{b}".strip("_") for a, b in raw.columns]
    raw = raw[pd.to_numeric(raw["observation_ID_obs"], errors="coerce").notna()].copy()
    raw["observation_ID_obs"] = raw["observation_ID_obs"].astype(int)
    raw["estimator_BC100"] = pd.to_numeric(raw["estimator_BC100"], errors="coerce")
    s2 = raw[raw["strategy_strategy"] == "S2"].merge(
        raw[raw["strategy_strategy"] == "S3"][["observation_ID_obs", "estimator_BC100"]]
        .rename(columns={"estimator_BC100": "BC100_S3"}), on="observation_ID_obs", how="left")
    s2["abs_diff_S2_S3"] = (s2["estimator_BC100"] - s2["BC100_S3"]).abs()
    s2 = s2[~s2["strategy_model"].astype(str).str.contains("singleExp", na=False)]
    s2 = s2[s2["abs_diff_S2_S3"].isna() | (s2["abs_diff_S2_S3"] <= MAX_DIFF_S2_S3)]
    return s2[s2["estimator_BC100"].between(*band)].copy()


fits = load_decay_fits(DECAY_FITS_XLSX, (DURABLE_MIN, DURABLE_MAX))
curves = [(r["strategy_model"], eval_s2_curve(r, YEARS)) for _, r in fits.iterrows()]
curves = [(m, c) for m, c in curves if not np.any(np.isnan(c))]
DECAY_CURVES = np.vstack([c for _, c in curves])
DECAY_MODELS = [m for m, _ in curves]
mean_curve = DECAY_CURVES.mean(axis=0)
E_KERNEL = np.clip(mean_curve[:-1] / 100 - mean_curve[1:] / 100, 0.0, 1.0)   # fraction lost in each year
print(f"{len(DECAY_CURVES)} decay fits, ensemble BC100 {mean_curve[100]:.1f} %")


def decay_CO2_at_year(t, A_CO2_stock, e_kernel):
    """CO2 released in year t by additions in earlier years (decay starts the year after)."""
    return sum(A_CO2_stock[y] * e_kernel[t - y - 1] for y in range(t) if t - y - 1 < len(e_kernel))


# ---- plantation cycle -----------------------------------------------------
def year_in_cycle(year):
    return None if year < 1 else (year - 1) % 25 + 1


def is_planting_year(year):
    return year_in_cycle(year) == 1


def is_stump_year(year):
    return year_in_cycle(year) == 25


def is_harvest_year(year):
    return year_in_cycle(year) in {3, 6, 9, 12, 15, 18, 21, 24}


def harvest_size(year):
    return FIRST_HARVEST_FACTOR if year_in_cycle(year) == 3 else 1.0


PLANTING_YEARS = [y for y in YEARS if is_planting_year(y)]
STUMP_YEARS = [y for y in YEARS if is_stump_year(y)]
HARVEST_YEARS = [y for y in YEARS if is_harvest_year(y)]
_w = {y: harvest_size(y) for y in HARVEST_YEARS}
harvest_weights = {y: w / sum(_w.values()) for y, w in _w.items()}

# ---- Brightway flows and activities ---------------------------------------
bio = bw2.Database("biosphere3")
CO2_FROM_STOCK = next(f for f in bio if f["name"] == "Carbon dioxide, from soil or biomass stock")
CO2_FOSSIL = next(f for f in bio if f["name"] == "Carbon dioxide, fossil")

TECH = {
    "electricity": (ECOINVENT_DB, "0e4b280caeeba40d5644b8d28328b0de"),   # market for electricity, medium voltage, SE
    "tap_water":   (ECOINVENT_DB, "4fe1f2dae4830593ee2d608cb2d5ff2c"),   # market for tap water, Europe without Switzerland
    "lorry":       (ECOINVENT_DB, "e76596f45bcb935639d775d8df91257e"),   # market for transport, freight, lorry >32 t, EURO5, RER
    "diesel":      (ECOINVENT_DB, "56d00f699d1086f7869005339a997aa5"),   # market for diesel, burned in agricultural machinery, GLO
}
ei = bw2.Database(ECOINVENT_DB)
for label, key in TECH.items():
    assert key in ei, f"{label} {key} not found in {ECOINVENT_DB}"


def add_technosphere(ds, key, amount, tag, unit=None):
    if amount:
        act = bw2.get_activity(key)
        ds["exchanges"].append({"type": "technosphere", "input": key, "amount": float(amount),
                                "name": act["name"], "unit": unit or act["unit"],
                                "unit_fu": unit or act["unit"], "tag": tag, "negative": False})


def add_biosphere(ds, key, amount, name, tag, negative=False):
    if amount:
        ds["exchanges"].append({"type": "biosphere", "input": key, "amount": float(amount), "name": name,
                                "unit": "kilogram", "unit_fu": "kilogram", "tag": tag, "negative": negative})


def add_biochar_chain(ds, biochar_kg):
    """Post-harvest chain for the biochar produced in a harvest year.
    Chip transport to the plant is the template row "Vägtransport, 60km"."""
    c = BIOCHAR_CHAIN
    add_technosphere(ds, TECH["diesel"], biochar_kg * c["diesel_chipping_kg"] * DIESEL_MJ_PER_KG,
                     "roadside_chipping_diesel", unit="megajoule")
    add_technosphere(ds, TECH["electricity"], biochar_kg * c["electricity_artificial_drying_kwh"],
                     "artificial_drying_electricity", unit="kilowatt hour")
    add_technosphere(ds, TECH["electricity"], biochar_kg * c["electricity_pyrolysis_kwh"],
                     "slow_pyrolysis_electricity", unit="kilowatt hour")
    add_technosphere(ds, TECH["tap_water"], biochar_kg * c["water_moistening_kg"],
                     "biochar_moistening_water", unit="kilogram")
    add_technosphere(ds, TECH["lorry"], biochar_kg * c["wet_biochar_15_out"] / 1000 * c["transport_distance_km"],
                     "transport_wet_biochar_to_field")
    add_technosphere(ds, TECH["diesel"], biochar_kg * c["diesel_soil_application_kg"] * DIESEL_MJ_PER_KG,
                     "soil_amendment_diesel", unit="megajoule")
    # heat credit, kg CO2-eq booked as avoided fossil CO2
    add_biosphere(ds, CO2_FOSSIL.key, -biochar_kg * c["avoided_district_heat_kwh"] * DH_FACTOR,
                  f"{CO2_FOSSIL['name']} (avoided district heating)", "avoided_district_heating", negative=True)


# ---- template, SOC change and zone means -----------------------------------
act_tpl = pd.read_excel(EXCEL_TEMPLATE, sheet_name="bw2lci", usecols="S:AE", nrows=1).iloc[0].to_dict()
exc_df = pd.read_excel(EXCEL_TEMPLATE, sheet_name="bw2lci", usecols="I:Q")

PLANTATION_TAGS = {"fg_Seedling", "fg_Pesticides"}
PLANTATION_NAMES = {"Plöjning", "Harvning"}
# "Skörd och flisning" (harvest with direct chipping) stands in for whole-rod harvesting;
# the rods are chipped again at the roadside, so diesel use is slightly overstated.
HARVEST_NAMES = {"Skörd och flisning", "Fälttransport", "Vägtransport, 60km"}
STUMP_NAMES = {"Sluttborttagning"}

luc = pd.read_excel(SALIX_ICBM_XLSX, index_col=0)
luc.index = [norm_zone(z) for z in luc.index]
luc.columns = [int(re.search(r"(\d+)$", str(c)).group(1)) for c in luc.columns]
luc_delta = luc.astype(float).diff(axis=1).fillna(0.0) * 1000 * 44 / 12    # t C -> kg CO2 per year

means_by_zone = {k: {norm_zone(z): v for z, v in d.items()} for k, d in zone_means.items()}
yield_by_zone = {norm_zone(z): safe_number(v) for z, v in zone_means_yield.items()}

# ---- activities -------------------------------------------------------------
new_data = {}
for zone in luc_delta.index:
    means = {k: safe_number(means_by_zone[k].get(zone)) for k in means_by_zone}
    biochar_total = yield_by_zone.get(zone, 0.0) * BIOCHAR_YIELD_ON_HARVEST * 1000 * 100   # kg over 100 years
    biochar_kg = np.zeros(101)
    co2_to_stock = np.zeros(101)
    for y in HARVEST_YEARS:
        biochar_kg[y] = biochar_total * harvest_weights[y]
        co2_to_stock[y] = biochar_kg[y] * CO2_PER_KG_BIOCHAR_TO_SOIL

    for yr in YEARS:
        code = f"willow_{zone}_y{yr:03d}"
        ds = dict(act_tpl)
        ds.update({"code": code,
                   "name": f"production and supply of willow woodchips, zone {zone}, year {yr}",
                   "exchanges": [{"input": (DATABASE_NAME, code), "output": (DATABASE_NAME, code),
                                  "type": "production", "amount": 1}]})
        annual = YEAR_FACTOR * (yr >= 1)

        for _, row in exc_df[exc_df.type == "technosphere"].iterrows():
            tag, name, base = row["tag"], row["name"], safe_number(row["amount"])
            if tag in PLANTATION_TAGS or name in PLANTATION_NAMES:
                if not is_planting_year(yr):
                    continue
                amt = (safe_number(means["Harvning"], base) if name == "Harvning" else base) / len(PLANTING_YEARS)
            elif name in HARVEST_NAMES:
                if not is_harvest_year(yr):
                    continue
                total = safe_number(means["Vägtransport"], base) if name == "Vägtransport, 60km" else base
                amt = total * harvest_weights[yr]
            elif name in STUMP_NAMES:
                if not is_stump_year(yr):
                    continue
                amt = base / len(STUMP_YEARS)
            else:
                amt = (safe_number(means["Nitrogen"], base) if name == "Nitrogen production" else base) * annual
            ds["exchanges"].append({"type": "technosphere", "name": name, "amount": amt,
                                    "unit": row["unit"], "unit_fu": row["unit_fu"],
                                    "input": eval(row["input"]),
                                    "negative": bool(row["negative"]) if not pd.isna(row["negative"]) else False,
                                    "tag": tag if isinstance(tag, str) else None})

        for _, row in exc_df[exc_df.type == "biosphere"].iterrows():
            tag, name = row["tag"], row["name"]
            if tag == "LUC_SOC_willow":
                amt = safe_number(luc_delta.at[zone, yr])
            elif tag == "fg_N2O_fertiliser_direct":
                amt = safe_number(means["Soil_N2O_direct"]) * annual
            elif tag == "fg_N2O_fertiliser_indirect":
                amt = safe_number(means["Soil_N2O_indirect"]) * annual
            elif tag == "LUC_N2O_litter":
                amt = safe_number(means["Soil_N2O_litter"]) * annual
            elif tag == "Biochar" or name == "Biochar":
                if not is_harvest_year(yr):
                    continue
                amt = co2_to_stock[yr]
            else:
                amt = safe_number(row["amount"]) * annual
            ds["exchanges"].append({"type": "biosphere", "tag": tag, "name": name, "amount": amt,
                                    "unit": row["unit"], "unit_fu": row["unit_fu"],
                                    "input": eval(row["input"]),
                                    "negative": bool(row["negative"]) if not pd.isna(row["negative"]) else False})

        if is_harvest_year(yr):
            add_biochar_chain(ds, biochar_kg[yr])

        ds["exchanges"].append({"type": "biosphere", "tag": "biochar_CO2_decay",
                                "name": f"{CO2_FROM_STOCK['name']} (biochar decay)",
                                "amount": decay_CO2_at_year(yr, co2_to_stock, E_KERNEL),
                                "unit": "kilogram", "unit_fu": "kg CO2/ha",
                                "input": CO2_FROM_STOCK.key, "negative": False})
        new_data[(DATABASE_NAME, code)] = ds

write_database(DATABASE_NAME, new_data)
print(f"{DATABASE_NAME}: {len(new_data)} activities ({len(luc_delta.index)} zones x 101 years)")

In [ ]:
# Decay curves used for the active scenario's durability band
fig, ax = plt.subplots(figsize=FIG_SIZE)
for model, curve in zip(DECAY_MODELS, DECAY_CURVES):
    ax.plot(YEARS, curve, lw=0.8, alpha=0.6, color="C0" if "doubleExp" in model else "C1")
ax.fill_between(YEARS, np.percentile(DECAY_CURVES, 5, axis=0), np.percentile(DECAY_CURVES, 95, axis=0),
                color="C0", alpha=0.15, label="5th–95th percentile")
ax.plot(YEARS, DECAY_CURVES.mean(axis=0), lw=2.5, color="C0", label="Ensemble mean")
ax.plot([], [], color="C0", lw=0.8, label="Double-exponential fits")
ax.plot([], [], color="C1", lw=0.8, label="Triple-exponential fits")
ax.set_xlabel("Year")
ax.set_ylabel("Carbon remaining (%)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.savefig(os.path.join(FIG_DIR, "fig04_decay_curves.png"))
plt.show()

## 7. Temperature response across zones

Yearly CO₂, CH₄ and N₂O of each zone, foreground and upstream, convolved with the
temperature response functions over 200 years. Results are kept with and without the heat
credit; the figure shows the result without it. The heat credit is in CO₂-eq and is
treated as a CO₂ pulse here.

In [ ]:
from bw2data.backends.peewee import ExchangeDataset

CHAR_HORIZON = 200
EMISSION_END = 100
KERNELS = generate_temp_responses(n=CHAR_HORIZON, vers="AR5", model="C2012")

db = bw2.Database(DATABASE_NAME)
selected = []
for act in db:
    m = re.match(r"willow_(.+)_y(\d+)$", act["code"])
    if m:
        selected.append((act, m.group(1), int(m.group(2))))
gas_inv = zone_year_gas_inventory(selected)

# heat credit per zone and year, so it can be removed
meta = {a["code"]: (z, y) for a, z, y in selected}
heat = []
for e in ExchangeDataset.select().where((ExchangeDataset.output_database == DATABASE_NAME)
                                        & (ExchangeDataset.type == "biosphere")):
    if e.data.get("tag") == "avoided_district_heating" and e.output_code in meta:
        zone, year = meta[e.output_code]
        gas, sign = gas_of_flow((e.input_database, e.input_code))
        heat.append((zone, year, gas, "heat_credit", sign * e.data["amount"]))
heat = pd.DataFrame(heat, columns=gas_inv.columns)
fg = gas_inv.source == "foreground"
gas_inv.loc[fg, "amount_kg"] -= (gas_inv[fg].merge(heat, on=["zone", "year", "flow"], how="left", suffixes=("", "_h"))
                                 .amount_kg_h.fillna(0.0).to_numpy())
gas_inv = pd.concat([gas_inv, heat], ignore_index=True)
print(gas_inv.groupby(["source", "flow"]).amount_kg.sum().unstack().round(0).to_string())


def temperature_by_zone(inv):
    out = []
    for zone, sub in inv.groupby("zone"):
        pulses = {g: np.zeros(EMISSION_END + 1) for g in GASES}
        for r in sub.itertuples():
            pulses[r.flow][r.year] += r.amount_kg
        dT = sum(convolve_pulses_with_kernel(pulses[g], KERNELS[g], CHAR_HORIZON) for g in GASES)
        out.append(pd.DataFrame({"zone": zone, "year": np.arange(CHAR_HORIZON + 1),
                                 "dT_total": dT, "CT_total": np.cumsum(dT)}))
    return pd.concat(out, ignore_index=True)


temp_with_heat = temperature_by_zone(gas_inv)
temp_no_heat = temperature_by_zone(gas_inv[gas_inv.source != "heat_credit"])
temp_with_heat.to_csv(os.path.join(OUT_DIR, f"temperature_{SCENARIO}_with_heat.csv"), index=False)
temp_no_heat.to_csv(os.path.join(OUT_DIR, f"temperature_{SCENARIO}_no_heat.csv"), index=False)

s = (temp_no_heat[temp_no_heat.year <= 100].groupby("year").dT_total
     .quantile([0.05, 0.25, 0.5, 0.75, 0.95]).unstack())
plt.figure(figsize=FIG_SIZE)
plt.fill_between(s.index, s[0.05], s[0.95], alpha=0.15, label="5th–95th percentile range across zones")
plt.fill_between(s.index, s[0.25], s[0.75], alpha=0.30, label="25th–75th percentile range across zones")
plt.plot(s.index, s[0.5], lw=2.5, label="Median across zones")
plt.axhline(0, lw=1)
plt.xlim(0, 100)
plt.xlabel("Year")
plt.ylabel("Global mean temperature response, ΔT (K)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.savefig(os.path.join(FIG_DIR, "fig07_temperature_response.png"))
plt.show()

ref = lambda d: d[(d.zone == REFERENCE_ZONE) & (d.year == 100)].dT_total.iloc[0]
print(f"Zone {REFERENCE_ZONE}, year 100: {ref(temp_no_heat):.3e} K without and "
      f"{ref(temp_with_heat):.3e} K with heat substitution")

## 8. GWP100 per zone

IPCC 2013 GWP100 of the 101 yearly activities of each zone, with and without the heat
credit, divided by 100 years for the per-year values.

In [ ]:
from rasterio.features import rasterize

UNIT_DIV = 100_000      # kg CO2-eq per ha over 100 years -> t CO2-eq per ha and year
CF = dict(bw2.Method(GWP_METHOD).load())


def score_database(db_name):
    """GWP100 per zone (kg CO2-eq/ha over years 0-100), with and without the heat credit."""
    zone_acts = {}
    for act in bw2.Database(db_name):
        m = re.match(r"willow_(.+)_y(\d+)$", act["code"])
        if m:
            zone_acts.setdefault(norm_zone(m.group(1)), []).append(act)
    rows, lca = [], None
    for zone, acts in zone_acts.items():
        fu = {a.key: 1 for a in acts}
        if lca is None:
            lca = bw2.LCA(fu, GWP_METHOD); lca.lci(); lca.lcia()
        else:
            lca.redo_lcia(fu)
        credit = sum(e["amount"] * CF.get(e["input"], 0.0)
                     for a in acts for e in a.biosphere() if e.get("tag") == "avoided_district_heating")
        rows.append((zone, lca.score, lca.score - credit))
    return pd.DataFrame(rows, columns=["zone", "with_heat", "without_heat"]).set_index("zone")


SCORES = {SCENARIO: score_database(DATABASE_NAME)}
gwp = SCORES[SCENARIO]
gwp.to_csv(os.path.join(OUT_DIR, f"gwp100_zones_{SCENARIO}.csv"))

v = gwp.without_heat / UNIT_DIV
studied_v = v[[int(z) <= STUDIED_MAX for z in v.index]]
closest = (studied_v - studied_v.median()).abs().idxmin()
print(f"Without heat substitution: {studied_v.min():.2f} to {studied_v.max():.2f} t CO2-eq/ha/yr, "
      f"median {studied_v.median():.2f}; zone closest to the median: {closest}")
print(f"Reference zone {REFERENCE_ZONE}: {v[REFERENCE_ZONE]:.3f} without, "
      f"{gwp.with_heat[REFERENCE_ZONE] / UNIT_DIV:.3f} with heat substitution")

# map, without heat substitution
with rasterio.open(YIELD_RASTER) as src:
    ymeta = src.meta.copy()
zmap = gpd.read_file(ZONES_SHP).to_crs(ymeta["crs"])
zmap = zmap[zmap["SKOMR"].astype(int) <= STUDIED_MAX]
grid = rasterize([(g, v.get(norm_zone(z), np.nan)) for g, z in zip(zmap.geometry, zmap["SKOMR"])],
                 out_shape=(ymeta["height"], ymeta["width"]), transform=ymeta["transform"],
                 fill=np.nan, all_touched=True, dtype=np.float32)
ymeta.update(dtype="float32", count=1, nodata=np.nan)
with rasterio.open(os.path.join(OUT_DIR, f"gwp100_map_no_heat_{SCENARIO}.tif"), "w", **ymeta) as dst:
    dst.write(grid, 1)

masked = np.ma.masked_invalid(grid)
ok = ~np.ma.getmaskarray(masked)
rr, cc = np.where(ok.any(axis=1))[0], np.where(ok.any(axis=0))[0]
crop = masked[max(rr[0] - 5, 0):rr[-1] + 6, max(cc[0] - 5, 0):cc[-1] + 6]
plt.figure(figsize=FIG_MAP)
im = plt.imshow(crop, cmap="gray")
plt.colorbar(im, fraction=0.046, pad=0.02, label="GWP100 (t CO$_2$-eq ha$^{-1}$ yr$^{-1}$)")
plt.axis("off")
plt.savefig(os.path.join(FIG_DIR, "fig10_gwp100_map.png"), pad_inches=0.05)
plt.show()

## 9. Contributions in the reference zone

GWP100 by component for one 25-year plantation, and the temperature response by
component over 100 years.

In [ ]:
from matplotlib.patches import Patch
from scipy.sparse.linalg import spsolve

CONTRIB_YEARS = list(range(1, 26))
CONTRIB_GROUPS = {        # Okabe-Ito colours
    "Biochar carbon storage": "#0072B2",
    "Biochar decay": "#56B4E9",
    "Soil organic carbon": "#E69F00",
    "Avoided emissions from heat substitution": "#009E73",
    "N$_2$O emissions": "#D55E00",
    "Fertiliser production": "#CC79A7",
    "Diesel and transport": "#4D4D4D",
    "Other inputs": "#BBBBBB",
}
N2O_TAGS = {"LUC_N2O_litter", "fg_N2O_fertiliser_direct", "fg_N2O_fertiliser_indirect"}


def contribution_group(tag, input_name):
    if tag == "Biochar":
        return "Biochar carbon storage"
    if tag == "biochar_CO2_decay":
        return "Biochar decay"
    if tag == "LUC_SOC_willow":
        return "Soil organic carbon"
    if tag == "avoided_district_heating":
        return "Avoided emissions from heat substitution"
    if tag in N2O_TAGS:
        return "N$_2$O emissions"
    n = (input_name or "").lower()
    if "fertiliser" in n:
        return "Fertiliser production"
    if "diesel" in n or "transport" in n:
        return "Diesel and transport"
    return "Other inputs"


acts = [bw2.get_activity((DATABASE_NAME, f"willow_{REFERENCE_ZONE}_y{y:03d}")) for y in CONTRIB_YEARS]
lca = bw2.LCA({acts[0].key: 1}, GWP_METHOD)
lca.load_lci_data(); lca.load_lcia_data()
unit_gwp = spsolve(lca.technosphere_matrix.T.tocsc(),
                   np.asarray((lca.characterization_matrix @ lca.biosphere_matrix).sum(axis=0)).ravel())
_, _, prod_row = lca_index_maps(lca)

rows = []
for yr, act in zip(CONTRIB_YEARS, acts):
    for e in act.exchanges():
        if e["type"] == "technosphere":
            inp = bw2.get_activity(e["input"])
            name, score, label = inp["name"], e["amount"] * unit_gwp[prod_row[e["input"]]], f"{inp['name']} ({inp['location']})"
        elif e["type"] == "biosphere":
            name, score, label = e.get("name", ""), e["amount"] * CF.get(e["input"], 0.0), e.get("tag") or e.get("name")
        else:
            continue
        rows.append(dict(year=yr, process=label, group=contribution_group(e.get("tag"), name), t_co2eq_ha=score / 1000))
contrib = pd.DataFrame(rows)

full = bw2.LCA({a.key: 1 for a in acts}, GWP_METHOD); full.lci(); full.lcia()
assert abs(contrib.t_co2eq_ha.sum() * 1000 - full.score) / abs(full.score) < 1e-6

contrib.pivot_table(index="process", columns="year", values="t_co2eq_ha", aggfunc="sum", fill_value=0.0) \
       .to_csv(os.path.join(OUT_DIR, f"contribution_by_process_{SCENARIO}_{REFERENCE_ZONE}.csv"))
grouped = (contrib.pivot_table(index="year", columns="group", values="t_co2eq_ha", aggfunc="sum", fill_value=0.0)
           .reindex(index=CONTRIB_YEARS, fill_value=0.0))
grouped = grouped[[g for g in CONTRIB_GROUPS if g in grouped.columns]]
grouped.to_csv(os.path.join(OUT_DIR, f"contribution_grouped_{SCENARIO}_{REFERENCE_ZONE}.csv"))

x = np.arange(len(CONTRIB_YEARS))
fig, ax = plt.subplots(figsize=FIG_SIZE)
pos, neg, handles = np.zeros(len(x)), np.zeros(len(x)), []
for g in grouped.columns:
    vals = grouped[g].to_numpy()
    p, n = np.clip(vals, 0, None), np.clip(vals, None, 0)
    if p.any():
        ax.bar(x, p, bottom=pos, color=CONTRIB_GROUPS[g], width=0.8); pos += p
    if n.any():
        ax.bar(x, n, bottom=neg, color=CONTRIB_GROUPS[g], width=0.8); neg += n
    if p.any() or n.any():
        handles.append(Patch(facecolor=CONTRIB_GROUPS[g], label=g))
ax.axhline(0, color="black", lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels([str(y) if y == 1 or y % 5 == 0 else "" for y in CONTRIB_YEARS])
ax.set_xlabel("Year")
ax.set_ylabel("Impact contribution (t CO$_2$-eq ha$^{-1}$)")
ax.grid(True, axis="y", alpha=0.3)
ax.legend(handles=handles, frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2)
plt.savefig(os.path.join(FIG_DIR, "fig09_gwp100_contributions.png"))
plt.show()

In [ ]:
from collections import defaultdict

YEARS_T = list(range(101))
acts = [bw2.get_activity((DATABASE_NAME, f"willow_{REFERENCE_ZONE}_y{y:03d}")) for y in YEARS_T]
lca = bw2.LCA({a.key: 1 for a in acts}); lca.lci()
_, _, prod_row = lca_index_maps(lca)
gas_unit = gas_per_unit_product(lca)

emis = defaultdict(lambda: {g: np.zeros(len(YEARS_T)) for g in GASES})
for yr, act in zip(YEARS_T, acts):
    for e in act.biosphere():
        gas, sign = gas_of_flow(e["input"])
        if gas:
            emis[grouped_tag(e.get("tag"), e.get("name", ""))][gas][yr] += e["amount"] * sign
    for e in act.technosphere():
        inp = bw2.get_activity(e["input"])
        group = grouped_tag(inp.get("tag"), inp.get("name", ""))
        for g in GASES:
            emis[group][g][yr] += gas_unit[g][prod_row[e["input"]]] * e["amount"]

curves = {grp: sum(convolve_pulses_with_kernel(t[g], KERNELS[g], CHAR_HORIZON) for g in GASES)
          for grp, t in emis.items()}
total_with = sum(curves.values())
total_without = total_with - curves.get("Avoided emissions from heat substitution", 0.0)

# the components add up to the zone's trajectories in section 7
for total, d in ((total_with, temp_with_heat), (total_without, temp_no_heat)):
    zref = d[d.zone == REFERENCE_ZONE].sort_values("year").dT_total.to_numpy()
    assert np.max(np.abs(total - zref)) / np.max(np.abs(zref)) < 1e-6

order = ["Biochar", "Soil organic carbon", "Willow production", "Avoided emissions from heat substitution"]
colors = {"Biochar": "#1f77b4", "Soil organic carbon": "#d62728",
          "Willow production": "#f7b6d2", "Avoided emissions from heat substitution": "#9edae5"}
yrs = np.arange(101)
plt.figure(figsize=FIG_SIZE)
for g in order:
    if g in curves:
        plt.plot(yrs, curves[g][:101], lw=2, color=colors[g], label=g)
plt.plot(yrs, total_with[:101], lw=3, color="black", label="Total, with heat substitution")
plt.plot(yrs, total_without[:101], lw=3, color="black", ls="--", label="Total, without heat substitution")
plt.axhline(0, color="grey", lw=1)
plt.xlim(0, 100)
plt.xlabel("Years after plantation establishment")
plt.ylabel("Global mean temperature response, ΔT (K)")
plt.grid(True, alpha=0.3)
plt.legend(loc="lower left", framealpha=0.9, fontsize="small", handlelength=1.8, labelspacing=0.3)
plt.savefig(os.path.join(FIG_DIR, "fig08_temperature_contributions.png"))
plt.show()

## 10. Other scenarios

Builds the databases of the other scenarios by re-running the cells tagged `config`,
`prep` and `inventory` from the notebook file, so save the notebook before running this
cell if you have edited those cells. Scenarios that share a yield raster share the soil
carbon run. The active scenario's raster is processed last, so the variables in memory
belong to `SCENARIO` again afterwards. Existing databases are skipped unless
`SKIP_EXISTING = False`.

In [ ]:
import json

SKIP_EXISTING = True


def tagged_sources(tag):
    nb = json.load(open(NOTEBOOK_FILE, encoding="utf-8"))
    return ["".join(c["source"]) for c in nb["cells"]
            if c["cell_type"] == "code" and tag in c.get("metadata", {}).get("tags", [])]


def run_tag(tag):
    sources = tagged_sources(tag)
    assert sources, f"no cells tagged {tag!r} in {NOTEBOOK_FILE}"
    for src in sources:
        exec(compile(src, f"<{tag}>", "exec"), globals())


def activate(scenario):
    g = globals()
    g["SCENARIO"] = scenario
    g["SCEN"] = SCENARIOS[scenario]
    g["YIELD_RASTER"] = os.path.join(DATA_DIR, SCENARIOS[scenario]["yield_raster"])
    g["DATABASE_NAME"] = scenario_db_name(scenario)
    g["SALIX_ICBM_XLSX"] = os.path.join(
        OUT_DIR, f"soc_willow_{os.path.splitext(SCENARIOS[scenario]['yield_raster'])[0]}.xlsx")


run_tag("config")
active = SCENARIO
groups = {}
for s in SCENARIOS:
    groups.setdefault(SCENARIOS[s]["yield_raster"], []).append(s)
order = [r for r in groups if r != SCENARIOS[active]["yield_raster"]] + [SCENARIOS[active]["yield_raster"]]
groups[SCENARIOS[active]["yield_raster"]] = [s for s in groups[SCENARIOS[active]["yield_raster"]] if s != active] + [active]

for raster in order:
    todo = [s for s in groups[raster] if not (SKIP_EXISTING and scenario_db_name(s) in bw2.databases)]
    if not todo and raster != SCENARIOS[active]["yield_raster"]:
        continue
    activate(groups[raster][0])
    print(f"Soil carbon and zone means for {raster}")
    run_tag("prep")
    for s in todo:
        activate(s)
        print(f"Inventory for {s}")
        run_tag("inventory")

activate(active)
for s in SCENARIOS:
    db = scenario_db_name(s)
    print(f"{s:24s} {db:32s} {len(bw2.Database(db)) if db in bw2.databases else 0:6d} activities")

## 11. Scenario comparison

In [ ]:
LABELS = {
    "high_yield": "High yield\n(base case)",
    "low_yield": "Low yield",
    "fossil_heat": "District heating\nreplacing fossil fuel",
    "low_biochar_stability": "Low biochar\nstability",
}
for s in SCENARIOS:
    if s not in SCORES:
        SCORES[s] = score_database(scenario_db_name(s))
    SCORES[s].to_csv(os.path.join(OUT_DIR, f"gwp100_zones_{s}.csv"))

table = pd.DataFrame({s: SCORES[s].loc[REFERENCE_ZONE] / UNIT_DIV for s in LABELS}).T
table.index.name = "scenario"
table["median_without_heat"] = [SCORES[s].without_heat.median() / UNIT_DIV for s in table.index]
table.to_csv(os.path.join(OUT_DIR, f"scenarios_zone_{REFERENCE_ZONE}.csv"))
print(table.round(2).to_string())

x = np.arange(len(table))
fig, ax = plt.subplots(figsize=FIG_WIDE)
b1 = ax.bar(x - 0.19, table.without_heat, 0.38, color="#bbbbbb", label="Without heat substitution")
b2 = ax.bar(x + 0.19, table.with_heat, 0.38, color="#4c72b0", label="With heat substitution")
ax.axhline(0, color="k", lw=0.8)
ax.set_xticks(x)
ax.set_xticklabels([LABELS[s] for s in table.index])
ax.set_ylabel("GWP100 (t CO$_2$-eq ha$^{-1}$ yr$^{-1}$)")
ax.bar_label(b1, fmt="%.2f", padding=3)
ax.bar_label(b2, fmt="%.2f", padding=3)
ax.legend()
plt.savefig(os.path.join(FIG_DIR, "fig11_scenarios.png"))
plt.show()

## 12. Monte Carlo and Sobol indices

For the reference zone. Rebuilding the database for every draw is not needed: the GWP100
of the zone is linear in its exchanges, so each uncertain parameter scales the exchanges it
drives. Soil carbon (ICBM) and biochar decay are not linear and are recomputed for each
draw. The checks in 12.1 make sure this reproduces the Brightway result.

Distributions and sources are listed in Table S13. The heat factor is not sampled: the main
output excludes heat substitution, and the result with heat adds the Swedish-average credit
to the same draws.

In [ ]:
# 12.1 Linear model of the reference zone
from scipy.sparse.linalg import spsolve

EMISSION_END = 100
YEARS_MC = np.arange(0, EMISSION_END + 1)
mc_acts = [bw2.get_activity((DATABASE_NAME, f"willow_{REFERENCE_ZONE}_y{y:03d}")) for y in YEARS_MC]

FEEDSTOCK_TAGS = {"roadside_chipping_diesel", "artificial_drying_electricity"}   # scale with the harvest
BIOCHAR_TAGS = {"biochar_moistening_water", "soil_amendment_diesel"}             # scale with the biochar


def mc_group(tag, name):
    name = name or ""
    if tag == "LUC_SOC_willow":                 return "soc"
    if tag == "biochar_CO2_decay":              return "decay"
    if tag == "Biochar" or name == "Biochar":   return "biochar_stock"
    if tag == "avoided_district_heating":       return "dh_credit"
    if tag == "slow_pyrolysis_electricity":     return "pyrolysis_electricity"
    if tag == "transport_wet_biochar_to_field": return "biochar_transport"
    if tag in FEEDSTOCK_TAGS:                   return "chain_feedstock"
    if tag in BIOCHAR_TAGS:                     return "chain_biochar"
    if name.startswith("Vägtransport"):         return "chips_transport"
    if name == "Nitrogen production":           return "n_fertiliser"
    if name == "Harvning":                      return "harrowing"
    if tag == "fg_N2O_fertiliser_direct":       return "n2o_direct"
    if tag == "fg_N2O_fertiliser_indirect":     return "n2o_indirect"
    if tag == "LUC_N2O_litter":                 return "n2o_litter"
    return "fixed"


MC_EXC = pd.DataFrame([dict(year=int(y), type=e["type"], tag=e.get("tag"), name=e.get("name"),
                            input=e["input"], amount=float(e["amount"]), group=mc_group(e.get("tag"), e.get("name")))
                       for y, a in zip(YEARS_MC, mc_acts) for e in a.exchanges() if e["type"] != "production"])
TECH_KEYS = sorted(MC_EXC.loc[MC_EXC.type == "technosphere", "input"].unique())


def unit_scores(lca_obj, keys):
    g = np.asarray((lca_obj.characterization_matrix @ lca_obj.biosphere_matrix).sum(axis=0)).ravel()
    y = spsolve(lca_obj.technosphere_matrix.T.tocsc(), g)
    return np.array([y[lca_obj.product_dict[k]] for k in keys])


lca = bw2.LCA({mc_acts[0].key: 1}, GWP_METHOD)
lca.load_lci_data(); lca.load_lcia_data()
U0 = unit_scores(lca, TECH_KEYS)
MC_EXC["gwp"] = [a * (U0[TECH_KEYS.index(k)] if t == "technosphere" else CF.get(k, 0.0))
                 for a, t, k in zip(MC_EXC.amount, MC_EXC.type, MC_EXC.input)]

MC_GROUPS = sorted(MC_EXC.group.unique())
TECH_AMT = {g: np.zeros(len(TECH_KEYS)) for g in MC_GROUPS}
BIO_GWP = {g: 0.0 for g in MC_GROUPS}
for r in MC_EXC.itertuples():
    if r.type == "technosphere":
        TECH_AMT[r.group][TECH_KEYS.index(r.input)] += r.amount
    else:
        BIO_GWP[r.group] += r.gwp

full = bw2.LCA({a.key: 1 for a in mc_acts}, GWP_METHOD); full.lci(); full.lcia()
MC_BASELINE = sum(TECH_AMT[g] @ U0 + BIO_GWP[g] for g in MC_GROUPS)
print(f"check: linear model {MC_BASELINE:,.1f} vs Brightway {full.score:,.1f} kg CO2-eq")
assert abs(MC_BASELINE - full.score) / abs(full.score) < 1e-6

biochar_kg = MC_EXC.loc[MC_EXC.tag == "slow_pyrolysis_electricity", "amount"].sum() / BIOCHAR_CHAIN["electricity_pyrolysis_kwh"]
assert abs(MC_EXC.loc[MC_EXC.group == "biochar_stock", "amount"].sum() / biochar_kg / CO2_PER_KG_BIOCHAR_TO_SOIL - 1) < 1e-3

# ICBM for the zone: spin-up plus 100 years of willow; arrays allowed, one value per draw
spin_zone = spin_in[spin_in["Region"].astype(int) == int(REFERENCE_ZONE)].iloc[0]
SPIN_IA, SPIN_IB = float(spin_zone["Ia"]), float(spin_zone["Ib"])
SALIX_FRAC = np.array([salix_input_fractions(y) for y in range(1, EMISSION_END + 1)])


def icbm_spinup(ia, ib, kY, kO, ha, hb, re, years=1000):
    eY, eO = np.exp(-kY * re), np.exp(-kO * re)
    Ya = Yb = Oa = Ob = 0.0
    for _ in range(years):
        Ya_in, Yb_in = Ya + ia, Yb + ib
        Oa = (Oa + ha * Ya_in * (1 - eY)) * eO
        Ob = (Ob + hb * Yb_in * (1 - eY)) * eO
        Ya, Yb = Ya_in * eY, Yb_in * eY
    return Ya, Yb, Oa + Ob


def icbm_soc(yield_eff, params=ICBM_PARAMS, full=False):
    """SOC change over 100 years (t C/ha), or the whole trajectory."""
    yield_eff = np.asarray(yield_eff, dtype=float)
    ya, yb, o = icbm_spinup(SPIN_IA, SPIN_IB, **params)
    ya, yb, o = ya + 0 * yield_eff, yb + 0 * yield_eff, o + 0 * yield_eff
    s0 = ya + yb + o
    eY, eO = np.exp(-params["kY"] * params["re"]), np.exp(-params["kO"] * params["re"])
    traj = [s0]
    for i_ag, i_bg in SALIX_FRAC:
        ya_in, yb_in = ya + i_ag * yield_eff, yb + i_bg * yield_eff
        o = (o + (params["hb"] * yb_in + params["ha"] * ya_in) * (1 - eY)) * eO
        ya, yb = ya_in * eY, yb_in * eY
        traj.append(ya + yb + o)
    return np.array(traj) if full else traj[-1] - s0


ref_pools = spinup[spinup.zone.astype(int) == int(REFERENCE_ZONE)].iloc[0]
assert np.allclose(icbm_spinup(SPIN_IA, SPIN_IB, **ICBM_PARAMS),
                   (ref_pools.Ya, ref_pools.Yb, ref_pools.O_above + ref_pools.O_below), rtol=1e-4)

# ICBM is linear in yield, so the zone's effective yield follows from the year-100 stock
soc_file = pd.read_excel(SALIX_ICBM_XLSX, index_col=0)
soc_file.index = [norm_zone(z) for z in soc_file.index]
SOC_FILE = soc_file.loc[REFERENCE_ZONE].to_numpy(dtype=float)
relax = icbm_soc(0.0, full=True)
YIELD_EFF = (SOC_FILE[-1] - relax[-1]) / (icbm_soc(1.0, full=True) - relax)[-1]
err = np.max(np.abs(icbm_soc(YIELD_EFF, full=True) - SOC_FILE)) / np.max(np.abs(SOC_FILE - SOC_FILE[0]))
print(f"check: ICBM trajectory of zone {REFERENCE_ZONE}, max error {err:.1%} of the 100-year change")
assert err < 0.01
SOC_DELTA0 = icbm_soc(YIELD_EFF)

# biochar decay: one curve per retained fit of the scenario's durability band
fits_mc = load_decay_fits(DECAY_FITS_XLSX, SCEN["durable"])
DECAY_CURVES_MC = np.vstack([c for c in (eval_s2_curve(r, YEARS_MC) for _, r in fits_mc.iterrows())
                             if not np.any(np.isnan(c))])
A_STOCK = (MC_EXC[MC_EXC.group == "biochar_stock"].groupby("year").amount.sum()
           .reindex(YEARS_MC, fill_value=0.0).to_numpy())


def decay_total(curve_pct):
    k = np.clip(curve_pct[:-1] / 100 - curve_pct[1:] / 100, 0.0, 1.0)
    cum = np.concatenate([[0.0], np.cumsum(k)])
    return sum(A_STOCK[y] * cum[min(EMISSION_END - y, len(k))] for y in YEARS_MC)


ens_total = decay_total(DECAY_CURVES_MC.mean(axis=0))
db_decay = MC_EXC.loc[MC_EXC.group == "decay", "amount"].sum()
print(f"check: decay {ens_total:,.1f} vs database {db_decay:,.1f} kg CO2 ({len(DECAY_CURVES_MC)} fits)")
assert abs(ens_total - db_decay) / db_decay < 1e-3
DECAY_RATIO = np.array([decay_total(c) for c in DECAY_CURVES_MC]) / ens_total

harvest = MC_EXC[(MC_EXC.type == "technosphere") & (MC_EXC.name == "Skörd och flisning")]
harvest_gwp = sum(a * U0[TECH_KEYS.index(k)] for a, k in zip(harvest.amount, harvest.input))
print(f"Harvest operation: {harvest_gwp / UNIT_DIV:.3f} t CO2-eq/ha/yr, "
      f"{abs(harvest_gwp / MC_BASELINE):.1%} of the zone result")

In [ ]:
# 12.2 ecoinvent background: GWP100 per unit of each input for 500 sampled sets of
# background matrices (pedigree uncertainty). Slow, so the draws are cached.
import time

N_BG, BG_SEED = 500, 42
BG_CACHE = os.path.join(OUT_DIR, f"mc_background_{SCENARIO}_{REFERENCE_ZONE}_{N_BG}_s{BG_SEED}.csv")
cols = [f"{d}|{c}" for d, c in TECH_KEYS]

if os.path.exists(BG_CACHE) and list(pd.read_csv(BG_CACHE, nrows=0).columns) == cols:
    BG_U = pd.read_csv(BG_CACHE).to_numpy()
else:
    mc = bw2.MonteCarloLCA({mc_acts[0].key: 1}, GWP_METHOD, seed=BG_SEED)
    mc.load_data()
    BG_U, t0 = [], time.time()
    while len(BG_U) < N_BG:
        mc.rebuild_technosphere_matrix(mc.tech_rng.next())
        mc.rebuild_biosphere_matrix(mc.bio_rng.next())
        mc.rebuild_characterization_matrix(mc.cf_rng.next())
        u = unit_scores(mc, TECH_KEYS)
        if np.all(np.isfinite(u)):
            BG_U.append(u)
        if len(BG_U) % 50 == 0:
            print(f"{len(BG_U)} draws, {time.time() - t0:.0f} s")
    BG_U = np.array(BG_U)
    pd.DataFrame(BG_U, columns=cols).to_csv(BG_CACHE, index=False)

bg_only = sum(BG_U @ TECH_AMT[g] + BIO_GWP[g] for g in MC_GROUPS) / UNIT_DIV
print(f"{len(BG_U)} draws. Background only: median {np.median(bg_only):.2f}, "
      f"95 % {np.percentile(bg_only, 2.5):.2f} to {np.percentile(bg_only, 97.5):.2f} t CO2-eq/ha/yr")

In [ ]:
# 12.3 Sampling and Sobol indices (Saltelli design; S1 after Saltelli et al. 2010, ST after Jansen 1999)
from scipy import stats
from scipy.stats import qmc

MC_LOG2_N, MC_SEED, N_REP = 13, 1, 10     # 8192 base samples, 10 independently scrambled replicates


def uniform(lo, hi):
    return stats.uniform(lo, hi - lo).ppf


def triang(lo, mode, hi):
    return stats.triang((mode - lo) / (hi - lo), lo, hi - lo).ppf


def index(n):
    return lambda u: np.minimum((np.asarray(u) * n).astype(int), n - 1)


def normal95(lo, mid, hi):
    """Normal with lo and hi at the 2.5th and 97.5th percentiles (split normal if
    asymmetric), truncated at zero."""
    sd_lo, sd_hi = (mid - lo) / 1.959964, (hi - mid) / 1.959964

    def ppf(u):
        z = stats.norm.ppf(np.clip(np.asarray(u, dtype=float), 1e-12, 1 - 1e-12))
        return np.maximum(mid + np.where(z < 0, sd_lo, sd_hi) * z, 0.0)
    return ppf


def indirect_n2o(n=200_000, seed=0):
    """Indirect N2O relative to its default: (FracGASF*EF4 + FracLEACH*EF5) with each
    component normal over its IPCC 2019 range (Table 11.3)."""
    u = np.random.default_rng(seed).random((4, n))
    ef4, ef5 = normal95(0.002, 0.010, 0.018)(u[0]), normal95(0.000, 0.011, 0.020)(u[1])
    gasf, leach = normal95(0.02, 0.11, 0.33)(u[2]), normal95(0.01, 0.24, 0.73)(u[3])
    ratio = np.sort((gasf * ef4 + leach * ef5) / (0.11 * 0.010 + 0.24 * 0.011))
    grid = (np.arange(n) + 0.5) / n
    return lambda q: np.interp(np.asarray(q), grid, ratio)


# Pyrolysis temperature moves mass yield and carbon content together, along the line from
# the design point (0.35, 0.72) to willow biochar at 489 C (0.28, 0.837; Hyvaluoma et al. 2018).
def biochar_c_from_mass(m):
    frac = (BIOCHAR_MASS_YIELD - np.asarray(m)) / (BIOCHAR_MASS_YIELD - 0.28)
    return BIOCHAR_C_CONTENT + frac * (0.837 - BIOCHAR_C_CONTENT)


ICBM_RANGES = {"kY": uniform(0.70, 0.90), "kO": uniform(0.006, 0.0102), "ha": uniform(0.13, 0.19),
               "hb": uniform(0.30, 0.44), "re": uniform(0.80, 1.20)}
for par, d in ICBM_RANGES.items():
    assert d(0.0) <= ICBM_PARAMS[par] <= d(1.0)

# name: (inverse CDF, label). Yield, N rate, emission factors, electricity and transport are
# multipliers on the deterministic value; the others are sampled as values.
MC_FACTORS = {
    "yield":              (triang(0.80, 1.0, 1.20), "Willow yield"),
    "n_rate":             (uniform(0.90, 1.30), "N fertiliser rate"),
    "ef_n2o_direct":      (normal95(0.20, 1.0, 1.80), "N$_2$O EF, direct"),
    "ef_n2o_indirect":    (indirect_n2o(), "N$_2$O EF, indirect"),
    "biochar_mass_yield": (triang(0.28, 0.35, 0.35), "Pyrolysis conditions (biochar yield and C)"),
    "handling_retention": (uniform(0.90, 1.00), "Biochar handling losses"),
    "decay_fit":          (index(len(DECAY_CURVES_MC)), "Biochar durability"),
    "pyro_elec":          (uniform(0.45, 1.00), "Pyrolysis electricity"),
    "transport":          (uniform(0.50, 1.50), "Transport distance"),
    "background":         (index(len(BG_U)), "ecoinvent background"),
}
TEX = {"kY": "k_Y", "kO": "k_O", "ha": "h_a", "hb": "h_b", "re": "r_e"}
for par, d in ICBM_RANGES.items():     # one draw per parameter, used for spin-up and willow alike
    MC_FACTORS[f"icbm_{par}"] = (d, f"ICBM ${TEX[par]}$")
FACTOR_NAMES = list(MC_FACTORS)
DH_DB = SCEN["dh_factor"]              # heat factor the database was built with


def mc_model(p, dh=0.0):
    """GWP100 of the zone (kg CO2-eq/ha, years 0-100); dh = displaced heat, kg CO2-eq/kWh."""
    Y = p["yield"]
    M = p["biochar_mass_yield"] / BIOCHAR_MASS_YIELD
    C = biochar_c_from_mass(p["biochar_mass_yield"])
    S = M * C * p["handling_retention"] / (BIOCHAR_C_CONTENT * HANDLING_RETENTION)
    U = BG_U[p["background"]]
    soc = icbm_soc(YIELD_EFF * Y, params=p["icbm"]) / SOC_DELTA0
    mult = {
        "fixed": 1.0,
        "harrowing": Y,
        "chips_transport": Y * p["transport"],
        "n_fertiliser": Y * p["n_rate"],
        "n2o_direct": Y * p["n_rate"] * p["ef_n2o_direct"],
        "n2o_indirect": Y * p["n_rate"] * p["ef_n2o_indirect"],
        "n2o_litter": Y * p["ef_n2o_direct"],
        "biochar_stock": Y * S,
        "decay": Y * S * DECAY_RATIO[p["decay_fit"]],
        "chain_feedstock": Y,
        "chain_biochar": Y * M,
        "pyrolysis_electricity": Y * p["pyro_elec"],
        "biochar_transport": Y * M * p["transport"],
        "dh_credit": Y * dh / DH_DB,   # heat export follows the feedstock
        "soc": soc,
    }
    return sum(mult[g] * (U @ TECH_AMT[g] + BIO_GWP[g]) for g in MC_GROUPS)


def params(u):
    p = {f: MC_FACTORS[f][0](u[:, j]) for j, f in enumerate(FACTOR_NAMES)}
    p["icbm"] = {par: p[f"icbm_{par}"] for par in ICBM_RANGES}
    return p


base = {f: np.array([1.0]) for f in ("yield", "n_rate", "ef_n2o_direct", "ef_n2o_indirect", "pyro_elec", "transport")}
base.update(biochar_mass_yield=np.array([BIOCHAR_MASS_YIELD]), handling_retention=np.array([HANDLING_RETENTION]),
            decay_fit=np.array([0]), background=np.array([0]), icbm=ICBM_PARAMS)
saved = BG_U, DECAY_RATIO
BG_U, DECAY_RATIO = U0[None, :], np.array([1.0])
assert abs(mc_model(base, dh=DH_DB)[0] - MC_BASELINE) / abs(MC_BASELINE) < 1e-6
BG_U, DECAY_RATIO = saved

k, N = len(FACTOR_NAMES), 2 ** MC_LOG2_N


def saltelli_run(seed):
    u = qmc.Sobol(d=2 * k, scramble=True, seed=seed).random(N)
    ua, ub = u[:, :k], u[:, k:]
    yab = np.empty((k, N))
    for j in range(k):
        uab = ua.copy(); uab[:, j] = ub[:, j]
        yab[j] = mc_model(params(uab))
    return ua, ub, mc_model(params(ua)), mc_model(params(ub)), yab


def sobol(ya, yb, yab):
    var = np.var(np.concatenate([ya, yb]), ddof=1)
    return np.mean(yb * (yab - ya), axis=1) / var, 0.5 * np.mean((ya - yab) ** 2, axis=1) / var


rep_S1, rep_ST = [], []
for r in range(N_REP):
    ua, ub, ya, yb, yab = saltelli_run(MC_SEED + r)
    if r == 0:
        UA, UB, YA, YB = ua, ub, ya, yb
    s1, st = sobol(ya, yb, yab)
    rep_S1.append(s1); rep_ST.append(st)
rep_S1, rep_ST = np.array(rep_S1), np.array(rep_ST)
S1, ST = rep_S1.mean(axis=0), rep_ST.mean(axis=0)
print(f"{N * (k + 2):,} model runs per replicate, {N_REP} replicates, {k} factors")

# output distribution from the A and B matrices of the first replicate
PA, PB = params(UA), params(UB)
Y_NO_HEAT = np.concatenate([YA, YB]) / UNIT_DIV
Y_HEAT = np.concatenate([mc_model(PA, dh=DH_SWEDISH_MIX), mc_model(PB, dh=DH_SWEDISH_MIX)]) / UNIT_DIV
credit_per_factor = (TECH_AMT["dh_credit"] @ U0 + BIO_GWP["dh_credit"]) / DH_DB
det_no_heat = (MC_BASELINE - credit_per_factor * DH_DB) / UNIT_DIV
det_heat = det_no_heat + credit_per_factor * DH_SWEDISH_MIX / UNIT_DIV


def interval(y):
    return np.median(y), *np.percentile(y, [2.5, 97.5])


summary = pd.DataFrame([("without heat substitution", det_no_heat, *interval(Y_NO_HEAT), np.mean(Y_NO_HEAT < 0)),
                        ("with heat substitution", det_heat, *interval(Y_HEAT), np.mean(Y_HEAT < 0))],
                       columns=["case", "deterministic", "median", "p2.5", "p97.5", "share_negative"])
sens = pd.DataFrame({"factor": FACTOR_NAMES, "label": [MC_FACTORS[f][1] for f in FACTOR_NAMES],
                     "S1": S1, "S1_spread": np.ptp(rep_S1, axis=0), "ST": ST, "ST_spread": np.ptp(rep_ST, axis=0)}
                    ).sort_values("ST", ascending=False).reset_index(drop=True)
summary.to_csv(os.path.join(OUT_DIR, f"mc_interval_{SCENARIO}_{REFERENCE_ZONE}.csv"), index=False)
sens.to_csv(os.path.join(OUT_DIR, f"mc_sobol_{SCENARIO}_{REFERENCE_ZONE}.csv"), index=False)
print(summary.round(3).to_string(index=False))
print(f"\nSobol indices, without heat substitution (sum S1 = {S1.sum():.2f})")
print(sens[["label", "S1", "S1_spread", "ST", "ST_spread"]].round(4).to_string(index=False))

In [ ]:
# Distribution of the result (A and B matrices of the first replicate)
fig, ax = plt.subplots(figsize=FIG_SIZE)
bins = np.linspace(min(Y_NO_HEAT.min(), Y_HEAT.min()), max(Y_NO_HEAT.max(), Y_HEAT.max()), 80)
for label, y, det, col in (("Without heat substitution", Y_NO_HEAT, det_no_heat, "C0"),
                           ("With heat substitution", Y_HEAT, det_heat, "C1")):
    med, lo, hi = interval(y)
    ax.hist(y, bins=bins, weights=np.full(len(y), 100.0 / len(y)), histtype="step", lw=1.6, color=col,
            label=f"{label}: {med:.2f} [{lo:.2f}, {hi:.2f}]")
    ax.axvline(lo, color=col, ls="--", lw=1)
    ax.axvline(hi, color=col, ls="--", lw=1)
    ax.axvline(det, color=col, lw=1.8)
ax.set_xlim(bins[0], 0)
ax.set_xlabel("GWP100 (t CO$_2$-eq ha$^{-1}$ yr$^{-1}$)")
ax.set_ylabel("Frequency (%)")
ax.legend(frameon=False, title="Median [95 % interval]; solid = deterministic, dashed = 2.5/97.5 %",
          loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=2)
plt.savefig(os.path.join(FIG_DIR, "fig12_monte_carlo.png"))
plt.show()

# Sobol indices; factors with ST < 0.5 % are pooled
s = sens.copy()
small = s.ST < 0.005
if small.sum() > 1:
    s = pd.concat([s[~small], pd.DataFrame([{"label": f"Other ({int(small.sum())} parameters)",
                                             "S1": s.loc[small, "S1"].sum(), "ST": s.loc[small, "ST"].sum()}])],
                  ignore_index=True)
s = s.iloc[::-1]
pos = np.arange(len(s))
fig, ax = plt.subplots(figsize=FIG_SIZE)
ax.barh(pos + 0.2, s.ST, 0.4, color="C0", label="Total order $S_T$")
ax.barh(pos - 0.2, s.S1.clip(lower=0), 0.4, color="C1", label="First order $S_1$")
ax.set_yticks(pos)
ax.set_yticklabels(s.label)
ax.set_xlabel("Share of output variance, without heat substitution")
ax.set_xlim(0, None)
ax.legend(frameon=False, loc="lower right")
plt.savefig(os.path.join(FIG_DIR, "fig13_sobol.png"))
plt.show()